# 2.11 · Bootstrap & Jackknife

> **课程定位**
> 第三条估计不确定性的路：**不要解析公式（频率）也不要先验（贝叶斯），用重抽样让数据自己说话**。中位数的 SE？相关系数的 CI？换任何怪统计量——同一套代码。这是 Efron 1979 的革命，也是"算力换数学"的第一个伟大案例。
> The third road: no formulas, no priors — resample and let the data speak. Efron's 1979 revolution; the first great compute-for-math trade.

> 💡 **面试相关**
> - "Bootstrap 的原理（一句话）" ★★★★
> - "怎么给中位数/AUC 算置信区间" ★★★★（没公式的统计量）
> - "Bootstrap 什么时候失效" ★★★
> - "Bagging 和 bootstrap 的关系" ★★★（Part 6 预告）

---

## 目录
1. [插件原理：世界 ≈ 样本 ⭐](#1)
2. [Bootstrap SE：通用配方](#2)
3. [三种 Bootstrap CI：percentile / basic / BCa ⭐](#3)
4. [覆盖率审计：BCa 为什么是默认](#4)
5. [无公式统计量：相关系数与 R²](#5)
6. [参数化 Bootstrap](#6)
7. [Jackknife：bootstrap 的前辈](#7)
8. [⚠ 失效场景](#8)
9. [实战：回归系数的 bootstrap（两种方案）](#9)
10. [小结](#10)


<a id="1"></a>
## 1. 插件原理：世界 ≈ 样本 ⭐ / The Plug-in Principle

理想世界：从总体 $F$ 反复抽样 → 看 $\hat\theta$ 的波动 → 得 SE。**问题：总体只给你一次抽样机会。**

**Efron 的暴论**：经验分布 $\hat{F}_n$（样本本身）是 $F$ 的最佳估计——**把"从总体抽样"替换成"从样本有放回重抽样"**：

$$F \to \hat{F}_n: \qquad x_1^*, \dots, x_n^* \overset{\text{有放回}}{\sim} \{x_1, \dots, x_n\}$$

每个 bootstrap 样本和原样本**同尺寸 $n$**（有放回 → 平均 63.2% 的原始点被抽中，其余重复——这个 63.2% 在 Part 6 bagging 的 OOB 里再相见）。
Same size n, with replacement — each resample contains ~63.2% of the original points. That 63.2% returns as OOB in bagging.


In [ ]:
import numpy as np
import scipy.stats as st
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

# "真实世界 vs bootstrap 世界" 的对照实验
# Real world vs bootstrap world, side by side
true_dist = st.lognorm(s=0.8, scale=np.exp(1))      # 总体 (上帝视角)
n = 150
sample = true_dist.rvs(n, random_state=7)            # 我们仅有的一份样本

B = 4000
# 上帝视角: 真的从总体反复抽 / God mode: resample the population
god_medians = np.median(true_dist.rvs((B, n), random_state=1), axis=1)
# 凡人视角: 从样本有放回重抽 / Mortal mode: resample the sample
boot_medians = np.median(rng.choice(sample, (B, n), replace=True), axis=1)

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.hist(god_medians, bins=70, density=True, alpha=0.6, label=f"true sampling dist (SE={god_medians.std():.3f})")
ax.hist(boot_medians, bins=70, density=True, alpha=0.6, label=f"bootstrap dist (SE={boot_medians.std():.3f})")
ax.legend(); ax.set_title("Median: the bootstrap world mimics the real world")
plt.tight_layout(); plt.show()


**两个直方图形状和宽度高度一致**——bootstrap 用一份样本复刻了"上帝才能看到"的抽样分布。中心略有偏移（样本中位数 ≠ 总体中位数），但**宽度（SE）对了**——而 SE 正是我们要的。
The widths match — and width (SE) is what we came for. The slight center shift just reflects that this sample's median isn't the population's.


<a id="2"></a>
## 2. Bootstrap SE：通用配方 / The Universal Recipe

```
for b in 1..B:                       # B = 1000~10000
    sample_b = resample(data, n, replace=True)
    theta_b  = statistic(sample_b)
SE = std(theta_1..theta_B)
```

**任何统计量**插进 `statistic` 都行——中位数、截尾均值、P95、Gini、AUC。这就是"插件"的含义。


In [ ]:
def boot_dist(data, statistic, B=4000, rng=rng):
    n = len(data)
    return np.array([statistic(rng.choice(data, n, replace=True)) for _ in range(B)])

# 一份偏态数据, 五个统计量的 SE — 同一套代码
# One skewed dataset, five statistics, one recipe
stats_zoo = {
    "mean":        np.mean,
    "median":      np.median,
    "P90":         lambda x: np.percentile(x, 90),
    "trimmed 10%": lambda x: st.trim_mean(x, 0.1),
    "Gini":        lambda x: np.abs(np.subtract.outer(x, x)).mean() / (2*x.mean()),
}
print(f"{'statistic':<13} {'estimate':>9} {'boot SE':>9}")
for name, f in stats_zoo.items():
    bd = boot_dist(sample, f, B=2000)
    print(f"{name:<13} {f(sample):>9.3f} {bd.std(ddof=1):>9.3f}")
print("\nGini 系数的解析 SE 公式要查论文; bootstrap 三行代码")


<a id="3"></a>
## 3. 三种 Bootstrap CI ⭐ / Three Bootstrap CIs

| 方法 | 区间 | 特点 |
|---|---|---|
| **Percentile** | $[\hat\theta^*_{(0.025)}, \hat\theta^*_{(0.975)}]$ | 最直观；偏态小样本下覆盖不足（2.5 已实测）|
| **Basic / pivotal** | $[2\hat\theta - \hat\theta^*_{(0.975)},\; 2\hat\theta - \hat\theta^*_{(0.025)}]$ | 把 bootstrap 分布"对折"；纠正部分偏差 |
| **BCa** ⭐ | percentile + 偏差校正 $z_0$ + 加速度 $a$（偏度校正）| **学界默认推荐**；scipy 内置 |

BCa 的两个校正量：
- $z_0$：bootstrap 分布有多少比例 < 原估计 → 量化**中位数偏差**
- $a$：用 jackknife（第 7 节）估计统计量的**偏度敏感性**


In [ ]:
from scipy.stats import bootstrap as scipy_bootstrap

data_tuple = (sample,)
print(f"统计量: median, 真值 = {true_dist.median():.3f}\n")
for method in ["percentile", "basic", "BCa"]:
    res = scipy_bootstrap(data_tuple, np.median, n_resamples=4000,
                          confidence_level=0.95, method=method, random_state=1)
    lo, hi = res.confidence_interval
    print(f"{method:<11}: [{lo:.3f}, {hi:.3f}]  宽 {hi-lo:.3f}")


<a id="4"></a>
## 4. 覆盖率审计：BCa 为什么是默认 / The Coverage Audit

用 2.5 的金标准（覆盖率模拟）正面对比——**偏态统计量 + 小样本**是分水岭：


In [ ]:
# 偏态场景: lognormal 的均值, n=30 / Skewed: lognormal mean at n=30
true_mean = true_dist.mean()
n_small, n_sim = 30, 800
cover = {"percentile": 0, "basic": 0, "BCa": 0}
for i in range(n_sim):
    s = true_dist.rvs(n_small, random_state=100+i)
    for method in cover:
        r = scipy_bootstrap((s,), np.mean, n_resamples=999,
                            method=method, random_state=i)
        lo, hi = r.confidence_interval
        cover[method] += (lo <= true_mean <= hi)

print(f"lognormal 均值, n={n_small}, 名义 95%:")
for m, c in cover.items():
    print(f"  {m:<11}: 实际覆盖 {c/n_sim:.1%}")


**BCa 把覆盖率从 ~88% 拉回最接近 95%**（仍不完美——重偏态小样本谁都难救，2.3 的教训）。**实践默认 BCa**，scipy 一个参数。
BCa pulls coverage closest to nominal. Default to it — it's one argument in scipy.


<a id="5"></a>
## 5. 无公式统计量：相关系数与 R² / Formula-free Statistics

bootstrap 真正的主场：**没有（好用的）解析 SE** 的量。注意二元数据要**成对重抽**（行抽样，保持 x-y 配对）：
Pairs must be resampled together — row resampling preserves the x-y coupling.


In [ ]:
# 相关系数的 bootstrap CI / CI for a correlation
n2 = 80
x = rng.normal(0, 1, n2)
y = 0.6*x + rng.normal(0, 0.8, n2)
r_hat = np.corrcoef(x, y)[0, 1]

idx_pool = np.arange(n2)
boot_r = np.array([
    np.corrcoef(x[idx], y[idx])[0, 1]
    for idx in [rng.choice(idx_pool, n2, replace=True) for _ in range(4000)]
])
lo, hi = np.percentile(boot_r, [2.5, 97.5])

print(f"r = {r_hat:.3f},  bootstrap 95% CI = [{lo:.3f}, {hi:.3f}]")
print(f"对照 Fisher z 变换解析 CI = "
      f"{tuple(np.round(np.tanh(np.arctanh(r_hat) + np.array([-1,1])*1.96/np.sqrt(n2-3)), 3))}")
print("两者接近 — 但 bootstrap 不需要知道 Fisher 变换的存在")


<a id="6"></a>
## 6. 参数化 Bootstrap / Parametric Bootstrap

变体：先拟合参数模型（2.2/2.9 的技能），**从拟合的模型生成**新样本而非重抽数据。

| | 非参 bootstrap | 参数化 bootstrap |
|---|---|---|
| 生成 | 重抽样本 | 从 $\hat{F}_{\text{model}}$ 模拟 |
| 假设 | 几乎没有 | 模型正确 |
| 小样本 | 离散性受限（只有 $n$ 个值可抽）| **更平滑**，外推到样本外的值 |
| 风险 | — | 模型错 → 全错 |

**适用**：n 很小（< 30）且对分布族有把握时；或统计量依赖尾部（样本里尾部点太少）。


In [ ]:
# n=15 的小样本: 两种 bootstrap 对比 / Tiny sample: both flavors
tiny = true_dist.rvs(15, random_state=3)

# 非参 / Nonparametric
np_se = boot_dist(tiny, np.median, B=4000).std(ddof=1)

# 参数化: 拟合 lognormal → 从模型生成 / Parametric: fit, then simulate
shape, loc, scale = st.lognorm.fit(tiny, floc=0)
pm = np.median(st.lognorm.rvs(shape, loc, scale, size=(4000, 15), random_state=4), axis=1)

print(f"中位数 SE:  非参 = {np_se:.3f},  参数化 = {pm.std(ddof=1):.3f}")
print(f"上帝视角真 SE (n=15) = "
      f"{np.median(true_dist.rvs((4000, 15), random_state=5), axis=1).std(ddof=1):.3f}")


<a id="7"></a>
## 7. Jackknife：bootstrap 的前辈 / The Jackknife

Quenouille (1949) / Tukey (1958)：**每次留一个出去**（leave-one-out），看统计量怎么变。

$$\hat\theta_{(i)} = \theta(\mathbf{x}_{-i}), \qquad \mathrm{SE}_{\text{jack}} = \sqrt{\frac{n-1}{n}\sum_i (\hat\theta_{(i)} - \bar{\hat\theta}_{(\cdot)})^2}$$

| | Jackknife | Bootstrap |
|---|---|---|
| 重复次数 | 恰好 $n$（确定性）| $B$（随机）|
| 适合 | **光滑统计量**（均值、方差、回归系数）| 几乎全部 |
| 不适合 | **非光滑**（中位数、分位数）⚠ 不一致！| 中位数也 OK |
| 现代角色 | BCa 的 $a$ 参数、影响函数诊断、LOO-CV 的祖先 | 主力 |


In [ ]:
def jackknife_se(data, statistic):
    n = len(data)
    theta_i = np.array([statistic(np.delete(data, i)) for i in range(n)])
    return np.sqrt((n-1)/n * np.sum((theta_i - theta_i.mean())**2))

# 光滑 (mean): jackknife 准 / Smooth: jackknife agrees
print(f"mean   SE:  解析 s/√n = {sample.std(ddof=1)/np.sqrt(len(sample)):.4f}, "
      f"jackknife = {jackknife_se(sample, np.mean):.4f}, "
      f"bootstrap = {boot_dist(sample, np.mean, 4000).std(ddof=1):.4f}")

# 非光滑 (median): jackknife 失灵 / Non-smooth: jackknife fails
print(f"median SE:  jackknife = {jackknife_se(sample, np.median):.4f}  ← 不可信!, "
      f"bootstrap = {boot_dist(sample, np.median, 4000).std(ddof=1):.4f}")
print("\n原因: 删一个点中位数大多不动, 偶尔跳一格 — 'all or nothing' 破坏方差估计")


<a id="8"></a>
## 8. ⚠ 失效场景 / When Bootstrap Fails

| 场景 | 为什么 | 替代 |
|---|---|---|
| **极值**（max/min）| 重抽样里 $\max^* \le \max$ 永远成立——分布退化 | 极值理论（2.3 的 Gumbel）/ 参数化 boot |
| **时间序列** | i.i.d. 重抽**打碎自相关** | block bootstrap（整块重抽，Part 14）|
| **极小样本**（n<10）| $\hat F_n$ 太粗糙 | 参数化 boot / 贝叶斯 |
| 无方差分布（Cauchy）| 2.3 的老朋友——SE 本身不存在 | 中位数 + 分位数法 |
| 依赖结构（聚类数据）| 同 cluster 行非独立 | cluster bootstrap（按群重抽）|


In [ ]:
# 极值失效演示 / The max failure, live
data_max = rng.uniform(0, 10, 200)
boot_max = boot_dist(data_max, np.max, B=4000)
print(f"样本 max = {data_max.max():.3f}")
print(f"bootstrap max 的取值: {len(np.unique(boot_max))} 个不同值, "
      f"{(boot_max == data_max.max()).mean():.0%} 的重抽直接等于样本 max")
print("→ bootstrap 分布退化成几根棍子, 完全没模拟出 max 的真实抽样分布")


<a id="9"></a>
## 9. 实战：回归系数的 bootstrap / Bootstrap for Regression

预告 Part 4 的主角。两种方案，**对应不同的世界观**：

| 方案 | 重抽什么 | 假设 |
|---|---|---|
| **Pairs bootstrap** | 整行 $(x_i, y_i)$ | x 也是随机的（观察性数据）⭐ 更稳 |
| **Residual bootstrap** | 拟合后的残差 | 模型正确 + x 固定（实验设计）|


In [ ]:
# 含异方差的回归数据 (经典公式会错, pairs bootstrap 仍对)
# Heteroscedastic data: classic SEs are wrong; pairs bootstrap copes
n3 = 200
x3 = rng.uniform(0, 10, n3)
y3 = 1.0 + 0.5*x3 + rng.normal(0, 0.3 + 0.25*x3, n3)    # 噪声随 x 增大!

def ols_slope(xx, yy):
    return np.polyfit(xx, yy, 1)[0]

slope_hat = ols_slope(x3, y3)

# 经典 OLS SE (假设同方差 — 在这数据上是错的)
resid = y3 - np.polyval(np.polyfit(x3, y3, 1), x3)
se_classic = np.sqrt(resid.var(ddof=2) / np.sum((x3 - x3.mean())**2))

# Pairs bootstrap
idx_pool = np.arange(n3)
boot_slopes = np.array([ols_slope(x3[i], y3[i])
                        for i in [rng.choice(idx_pool, n3, replace=True) for _ in range(4000)]])

# 上帝视角: 真 SE / God-mode truth
god_slopes = []
for s in range(2000):
    r2 = np.random.default_rng(1000+s)
    xg = r2.uniform(0, 10, n3)
    yg = 1.0 + 0.5*xg + r2.normal(0, 0.3 + 0.25*xg, n3)
    god_slopes.append(ols_slope(xg, yg))

print(f"slope = {slope_hat:.4f} (真值 0.5)")
print(f"经典 OLS SE        = {se_classic:.4f}   ← 异方差下偏小")
print(f"pairs bootstrap SE = {boot_slopes.std(ddof=1):.4f}")
print(f"上帝视角真 SE       = {np.std(god_slopes, ddof=1):.4f}   ← bootstrap 对得多")


**经典 SE 在异方差下系统性偏小**（假阳风险），pairs bootstrap 贴住真值——**不知道误差结构时的安全网**。这也是计量经济学"稳健标准误"思想的近亲（Part 4.2 再会）。
Classic SEs understate under heteroscedasticity; pairs bootstrap nails it — the safety net when you don't know the error structure.


<a id="10"></a>
## 10. 小结 / Summary

```
插件原理: F ≈ F̂ₙ → 有放回重抽 n 个 × B 次 → 统计量的经验分布
  SE = std(θ*₁..θ*_B)
  CI: percentile < basic < BCa ⭐ (偏差+偏度双校正, scipy 默认可选)

变体: 参数化 boot (小样本+信模型) / block boot (时序) / cluster boot (分组)
Jackknife: 留一法, 光滑统计量 OK, 中位数失灵; 活在 BCa 的 a 里

失效: 极值(退化) / 时序(打碎相关) / 无方差(Cauchy) / n<10
回归: pairs boot 抗异方差 — 不知误差结构时的默认
```

### 💡 面试速查
1. **一句话原理**："样本是总体的最佳替身，重抽样模拟重新做实验"
2. **63.2%**：每次重抽含的原始点比例（→ bagging OOB）
3. **AUC/中位数/Gini 的 CI** → bootstrap 三行
4. **极值不能 bootstrap**（max* ≤ max 恒成立 → 退化）
5. **时序要 block bootstrap**
6. **BCa 是默认推荐**

### 下一节
**2.12 蒙特卡洛**——Part 2 收官：bootstrap 只是它的特例。积分、π、重要性采样、逆变换/拒绝采样、MCMC 预告。
